# Patient Specific Model

In [67]:
from pathlib import Path
import numpy as np
import h5py

import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, average_precision_score


In [68]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, GridSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score


# 1. Separate Records

> Create a dictionary with the all the data together

In [76]:

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("Features/Seizure").glob("chb24_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

non_files = sorted(
    Path("Features/Non_Seizure").glob("chb24_*_features.mat")
)

for file in non_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]

    if X.shape[0] == 1530:
        X = X.T

    y = np.zeros(X.shape[0], dtype=int)

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(
        "[NON-SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

print("\nTotal records:", len(records))

[SEIZURE] chb24_01 (1437, 1530) (1437,) positives = 11
[SEIZURE] chb24_03 (1437, 1530) (1437,) positives = 12
[SEIZURE] chb24_04 (1437, 1530) (1437,) positives = 13
[SEIZURE] chb24_06 (1437, 1530) (1437,) positives = 10
[SEIZURE] chb24_07 (1437, 1530) (1437,) positives = 9
[SEIZURE] chb24_09 (1437, 1530) (1437,) positives = 8
[SEIZURE] chb24_11 (1437, 1530) (1437,) positives = 28
[SEIZURE] chb24_13 (1437, 1530) (1437,) positives = 6
[SEIZURE] chb24_14 (1437, 1530) (1437,) positives = 11
[SEIZURE] chb24_15 (1437, 1530) (1437,) positives = 7
[SEIZURE] chb24_17 (1437, 1530) (1437,) positives = 27
[SEIZURE] chb24_21 (1437, 1530) (1437,) positives = 27
[NON-SEIZURE] chb24_02 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb24_05 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb24_08 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb24_10 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb24_12 (1917, 1530) (1917,) positives = 0
[NON-SEIZURE] chb24_16 (1917, 1530) (1917,) positives 

>records = { <br>
    "chb01_03": (X_03, y_03),<br>
    "chb01_04": (X_04, y_04),<br>
    "chb01_05": (X_05, y_05),<br>
    ...<br>
}

## 1.1  Separate Seizure and Non-seizure

> Separatre the dictionary in 2

In [70]:
# =====================================================
# SPLIT SEIZURE / NON-SEIZURE RECORDS
# =====================================================
seizure_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) > 0
}

non_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) == 0
}


print("Seizure:", len(seizure_records))
print("Non seizure:", len(non_records))

Seizure: 0
Non seizure: 0


## 1.2 Pick $R$ Non-seizure Records

In [71]:
# =====================================================
# SELECT NON-SEIZURE RECORDS FOR TRAINING
# =====================================================

r = 0
random.seed(42)


selected_non = random.sample(
    list(non_records.keys()),
    r
)


print("Selected non seizure:")
print(selected_non)

Selected non seizure:
[]


# 2. Training Function

> This will remove the test record from train

In [72]:
def create_train_test(test_record):

    X_train = []
    y_train = []
    train_groups = []


    # -------------------------
    # seizure records
    # -------------------------

    for name, (X,y) in seizure_records.items():

        if name == test_record:
            continue


        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # -------------------------
    # non seizure records
    # -------------------------

    for name in selected_non:

        X,y = non_records[name]

        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # combine

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    train_groups = np.array(train_groups)



    # test

    X_test, y_test = seizure_records[test_record]


    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups
    )

# 3. Model

> Things that we want to tune:</br>
> - k number of features </br>
> - Value of C

In [73]:
from joblib import Memory


In [74]:
pip install timescoring

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, LeaveOneGroupOut, cross_val_predict,
)
from sklearn.metrics import make_scorer
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 8               # clinical cap on persistence (latency control)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 5       # a-priori ceiling. NOTE: Shoeb is 0.08 FD/h ~= 1.9 FP/day;
                              # set this on a principled basis (e.g. Shoeb-matched), not to flatter results.
SELECT_FPR        = 0.01      # inner-CV selection: cap on segment-level FPR for the proxy scorer

OUTER_N_JOBS = 8              # parallelism over LORO folds (inner stays at 1)

PATIENT = os.environ.get("PATIENT", "chb04")   # filename prefix; set per run e.g. PATIENT=chb01

# --- SzCORE event-scoring parameters, pinned explicitly (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# percentile (feature selection), C, and gamma are all tuned via the grid below
param_grid = {
    "select__percentile": [5, 10, 15],
    "svm__C": [ 0.01],
    "svm__gamma": ["scale"],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", class_weight="balanced",
                    cache_size=8000)),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

def _arming_times(sc, thr, k):
    # End time of the k-th consecutive positive window (the alarm-defining window),
    # one per run that reaches k. This is the latency convention used throughout.
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm

def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat

def _safe_div(a, b):
    return a / b if b else float("nan")

# --- SEGMENT-LEVEL (window-level) confusion-matrix metrics at a given threshold ---
# Thresholds the raw window scores (no k-persistence; that is event-level logic).
# Called with thr=0.0 (the natural decision_function boundary).
# Returns accuracy, specificity, sensitivity, precision, f1 and the raw tp/fp/fn counts.
def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn

# --- HYPERPARAMETER-SELECTION SCORER (inner CV) ---------------------------
# Maximize segment-level recall subject to a cap on segment-level FPR: selects
# hyperparameters at the low-false-alarm operating point we deploy at.
def recall_at_fpr(y_true, y_score, target_fpr=SELECT_FPR):
    y_true = np.asarray(y_true).astype(int)
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return 0.0
    order = np.argsort(-y_score)
    ys = y_true[order]
    P = ys.sum(); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    fpr = fp / N; recall = tp / P
    feasible = recall[fpr <= target_fpr]
    return float(feasible.max()) if feasible.size else 0.0

import sklearn
from packaging import version as _v
if _v.parse(sklearn.__version__) >= _v.parse("1.4"):
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, response_method="decision_function")
else:
    EVENT_PROXY_SCORER = make_scorer(recall_at_fpr, needs_threshold=True)

def alarm_rows_of(record, s, sc, thr, k):
    # Every alarm (hypothesis event), flagged true/false exactly as timescoring
    # counts s.tp / s.fp (overlap with tpMask = ref events, extended by 30/60s
    # tolerance, that were detected). For true alarms, latency = arming time
    # (end of k-th consecutive window) inside the alarm and inside the detected
    # seizure's [onset-30, offset+60] window, minus that seizure's onset.
    # #false-alarms == s.fp by construction.
    TOL_PRE, TOL_POST = 30, 60
    fs = s.fs
    arm_times = _arming_times(sc, thr, k)
    seiz = [(float(r["start_sec"]), float(r["end_sec"]))
            for _, r in SEIZ[SEIZ["file"] == record].iterrows()]
    rows, aid = [], 0
    for hs, he in s.hyp.events:
        a, b = round(hs * fs), round(he * fs)
        is_true = bool(np.any(s.tpMask[a:b]))
        latency = float("nan")
        if is_true:
            cands = []
            for onset, offset in seiz:
                if hs <= (offset + TOL_POST) and (onset - TOL_PRE) <= he:  # extended overlap
                    arms_in = [t for t in arm_times
                               if (onset - TOL_PRE) <= t <= (offset + TOL_POST)
                               and (hs - 1e-6) <= t <= (he + 1e-6)]
                    if arms_in:
                        cands.append(min(arms_in) - onset)
            if cands:
                latency = float(min(cands))
        rows.append(dict(record=record, alarm_id=aid,
                         start_sec=float(hs), end_sec=float(he),
                         duration_sec=float(he - hs),
                         alarm_type="true" if is_true else "false",
                         latency=latency))
        aid += 1
    return rows

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 25)))
    else:
        thr_grid = THRESH_GRID

    best = None
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=len(np.unique(groups)))
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring=EVENT_PROXY_SCORER, cv=inner, n_jobs=1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def n_selected_of(model):
    return int(model.named_steps["select"].get_support().sum())

def oof_train_record_scores(best_params, exclude):
    Xtr, ytr, grps = build_train(exclude=exclude)
    hp = {p: best_params[p] for p in ("select__percentile", "svm__C","svm__gamma")}
    est = make_pipe().set_params(**hp)
    oof_scores = cross_val_predict(
        est, Xtr, ytr, groups=grps,
        cv=LeaveOneGroupOut(), method="decision_function", n_jobs=1,
    )
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        y_name = ytr[m]
        out.append((name, oof_scores[m], y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO  (outer folds parallelized; inner stays single-threaded)
# =====================================================
def process_seizure_fold(test_record):
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, _ = tune_fit(Xtr, ytr, grps)
    oof = oof_train_record_scores(best, exclude=test_record)
    thr, k = calibrate(oof)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, best=best, n_selected=n_selected_of(model))

results = Parallel(n_jobs=OUTER_N_JOBS)(
    delayed(process_seizure_fold)(r) for r in seizure_records.keys()
)

test_outputs = []
fold_choices = []
meta = {}
for res in results:
    test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                         res["thr"], res["k"]))
    fold_choices.append((res["record"], res["thr"], res["k"]))
    meta[res["record"]] = dict(type="seizure", C=res["best"]["svm__C"],
                               gamma=res["best"]["svm__gamma"],
                               selected_features=res["n_selected"])
    print(f"[SEIZ] {res['record']}  params={res['best']}  "
          f"thr={res['thr']:.3f} k={res['k']}  test_pos={int(res['y'].sum())}")

# Non-model: one global threshold/k for all seizure-free records
Xall, yall, gall = build_train(exclude=None)
full_model, full_best, _ = tune_fit(Xall, yall, gall)
full_n_selected = n_selected_of(full_model)
oof_full = oof_train_record_scores(full_best, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
print(f"[NON-MODEL] params={full_best}  thr={thr_fp:.3f} k={k_fp}  (scores all seizure-free records)")
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    meta[name] = dict(type="nonseizure", C=full_best["svm__C"],
                      gamma=full_best["svm__gamma"],
                      selected_features=full_n_selected)

# =====================================================
# PER-RECORD pass -> segment-level table + alarm log + event totals
# Segment ratio metrics are only meaningful on seizure records; on seizure-free
# records they are set to NaN (raw tp/fp/fn are still reported, since those are
# honest counts and the fp count is the false-alarm contribution).
# =====================================================
TP = FP = FN = 0
all_latencies = []
segment_rows = []
alarm_rows = []
for record, sc, y, dur, thr, k in test_outputs:
    is_seiz = meta[record]["type"] == "seizure"

    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS),
                             Annotation(hyp, GRID_FS), SZCORE_PARAMS)
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp   # non-seizure FPs count toward FP/day

    alarm_rows.extend(alarm_rows_of(record, s, sc, thr, k))
    if is_seiz:
        all_latencies.extend(detection_latencies(record, sc, thr, k))

    seg_acc, seg_spec, seg_sens, seg_prec, seg_f1, seg_tp, seg_fp, seg_fn = \
        segment_metrics(y, sc, 0.0)
    if not is_seiz:
        seg_acc = seg_spec = seg_sens = seg_prec = seg_f1 = np.nan

    m = meta[record]
    segment_rows.append({
        "patient": PATIENT,
        "record": record,
        "n_windows": int(len(y)),
        "n_seizures_windows": int(np.asarray(y).sum()),
        "record_length": dur,
        "C": m["C"], "gamma": m["gamma"],
        "selected_features": m["selected_features"],
        "threshold": thr, "k": k,
        "seg_accuracy": seg_acc,
        "seg_specificity": seg_spec,
        "seg_sensitivity": seg_sens,
        "seg_precision": seg_prec,
        "seg_f1": seg_f1,
        "tp": seg_tp, "fp": seg_fp, "fn": seg_fn,
    })

# event-level totals
sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
total_len_sec = sum(t[3] for t in test_outputs)
days = total_len_sec / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Total length  : {total_len_sec/3600:.2f} h")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, kk in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={kk}")

# =====================================================
# SAVE
#   1) SEGMENT_PER_RECORD_<PATIENT>.csv   segment-level metrics per record
#   2) <PATIENT>_ALARMS.csv               every alarm, true/false + latency
#   3) RESULTS_SUMMARY.csv                one appended row per patient
# =====================================================
segment_cols = ["patient", "record", "n_windows", "n_seizures_windows",
                "record_length", "C", "gamma", "selected_features",
                "threshold", "k",
                "seg_accuracy", "seg_specificity", "seg_sensitivity",
                "seg_precision", "seg_f1", "tp", "fp", "fn"]
pd.DataFrame(segment_rows, columns=segment_cols) \
  .to_csv(f"SEGMENT_PER_RECORD_{PATIENT}.csv", index=False)

alarm_cols = ["record", "alarm_id", "start_sec", "end_sec", "duration_sec",
              "alarm_type", "latency"]
pd.DataFrame(alarm_rows, columns=alarm_cols) \
  .to_csv(f"{PATIENT}_ALARMS.csv", index=False)

summary = {
    "patient": PATIENT,
    "total_length_sec": total_len_sec,
    "total_length_hours": total_len_sec / 3600,
    "TP": TP, "FP": FP, "FN": FN,
    "sensitivity": sens, "precision": prec, "f1": f1,
    "FP/day": FP / days,
    "FP/hour": FP / (days * 24),
    "latency_median": med_lat,
    "n_detected": len(all_latencies),
}
master = "RESULTS_SUMMARY.csv"
pd.DataFrame([summary]).to_csv(
    master, mode="a", header=not os.path.exists(master), index=False,
)

print(f"\nSaved: SEGMENT_PER_RECORD_{PATIENT}.csv, {PATIENT}_ALARMS.csv, "
      f"appended row to {master}")

KeyboardInterrupt: 

In [78]:
from pathlib import Path
import h5py
import numpy as np

feature_dir = Path("Features")

bad_files = []

for file in sorted(feature_dir.rglob("*.mat")):

    try:
        with h5py.File(file, "r") as f:
            X = f["all_feats"][:]

        problems = []

        # MATLAB complex stored as structured dtype
        if X.dtype.names is not None:
            problems.append(f"structured dtype {X.dtype}")

        # normal numpy complex
        if np.iscomplexobj(X):
            problems.append("complex")

        # check NaN/Inf
        if X.dtype.names is None:
            if np.isnan(X).any():
                problems.append("NaN")

            if np.isinf(X).any():
                problems.append("Inf")

        # check dimensions
        if X.ndim != 2:
            problems.append(f"wrong ndim {X.ndim}")

        # expected features
        if X.ndim == 2:
            if X.shape[0] != 1530 and X.shape[1] != 1530:
                problems.append(f"wrong feature size {X.shape}")

        if problems:
            bad_files.append((file.name, X.shape, problems))

    except Exception as e:
        bad_files.append((file.name, "LOAD ERROR", str(e)))


print("\n===== PROBLEM FILES =====")

if not bad_files:
    print("No problems found ✅")
else:
    for name, shape, issue in bad_files:
        print("\n", name)
        print("shape:", shape)
        print("issue:", issue)


===== PROBLEM FILES =====

 chb04_12_features.mat
shape: LOAD ERROR
issue: Unable to synchronously open file (truncated file: eof = 89128448, sblock->base_addr = 512, stored_eof = 95321548)

 chb22_11_features.mat
shape: (1922, 1530)
issue: ["structured dtype [('real', '<f8'), ('imag', '<f8')]"]


In [82]:
from pathlib import Path
import h5py, numpy as np

for f in sorted(Path("Features/Non_Seizure").glob("*_features.mat")):
    with h5py.File(f, "r") as h:
        X = h["all_feats"][:]
    # h5py reads MATLAB complex as a compound dtype with 'real'/'imag' fields
    if X.dtype.names and "imag" in X.dtype.names:
        imag = np.abs(X["imag"])
        real = np.abs(X["real"])
        print(f"{f.name}: max|imag|={imag.max():.2e}  "
              f"max|imag/real|={(imag/(real+1e-12)).max():.2e}")

chb22_11_features.mat: max|imag|=4.49e-01  max|imag/real|=1.21e+00


In [84]:
from pathlib import Path
import h5py, numpy as np

f = Path("Features/Non_Seizure/chb04_12_features.mat")   # adjust dir if needed
with h5py.File(f, "r") as h:
    X = h["all_feats"][:]

# h5py reads MATLAB complex as compound dtype; orient so rows = windows
if X.dtype.names and "imag" in X.dtype.names:
    real = X["real"]; imag = X["imag"]
else:
    real = X; imag = np.zeros_like(X)

# make rows = windows (N_FEATURES on the other axis)
if real.shape[0] == 1530:
    real, imag = real.T, imag.T

# per-window worst imaginary magnitude
per_window_imag = np.abs(imag).max(axis=1)
bad = per_window_imag > 1e-6

print(f"shape (windows x feats): {real.shape}")
print(f"contaminated windows: {bad.sum()} / {len(bad)} "
      f"({100*bad.mean():.1f}%)")
print(f"bad window indices: {np.where(bad)[0]}")

shape (windows x feats): (7685, 1530)
contaminated windows: 2 / 7685 (0.0%)
bad window indices: [ 95 135]
